# Schedule Generation

## Enterprise Interest Rate Swap Pricing Platform

## Introduction

An Interest Rate Swap (IRS) is fundamentally a contract that exchanges a sequence of future cash flows between two counterparties.

Although a trade specifies the contractual terms such as the effective date, maturity date, payment frequency, fixed rate, and floating reference index, it does not explicitly define every payment period. Before any coupon can be calculated or discounted, the pricing engine must first determine **when** each payment occurs.

This process is known as **schedule generation**.

Schedule generation converts the high-level contractual description of a trade into a complete sequence of contractual payment periods. These payment periods become the foundation for every subsequent stage of the pricing process, including coupon generation, discounting, present value calculation, and risk measurement.

In production pricing libraries, schedule generation is therefore one of the earliest components implemented after trade representation.

## Objectives

In this notebook we will:

- Understand the role of schedule generation within an Interest Rate Swap pricing engine.
- Explain the financial concepts behind payment schedules.
- Design immutable domain objects to represent contractual payment periods.
- Implement a schedule generation service.
- Generate payment schedules for vanilla Interest Rate Swaps.
- Demonstrate the generated schedule using a complete trade example.

By the end of this notebook, the pricing engine will be capable of transforming a trade definition into a sequence of contractual payment periods that can be consumed by future pricing modules.

## Position within the Pricing Engine

Building a pricing library is an incremental process. Each module provides the foundation for the next.

The current project follows the workflow below.

Schedule generation occupies an important position in this workflow.

The yield curve determines how future cash flows will be discounted, but before discounting is possible, the engine must first identify every contractual payment date associated with the trade.

For this reason, schedule generation acts as the bridge between contractual trade representation and financial valuation.

## Why is Schedule Generation Necessary?

A trade description contains only the contractual parameters of an Interest Rate Swap.

For example:

- Effective Date
- Maturity Date
- Payment Frequency
- Fixed Rate
- Floating Reference Index

However, the pricing engine cannot calculate coupon payments directly from this information.

Instead, it must derive every accrual period defined by the contract.

For an annual five-year swap, this means constructing five distinct payment periods.

Each payment period stores:

- Accrual Start Date
- Accrual End Date
- Payment Date
- Accrual Length

These payment periods are subsequently used to generate coupon cash flows and calculate present values.

## Financial Concepts

Before implementing the schedule generation engine, it is important to understand the contractual dates that define an Interest Rate Swap.

### Effective Date

The effective date represents the start of the contractual agreement. Interest begins accruing from this date, although no cash is exchanged when the swap becomes effective.

### Maturity Date

The maturity date marks the end of the swap contract. The final coupon is calculated up to this date, after which the contract terminates.

### Payment Frequency

Payment frequency specifies how often coupon payments are exchanged between the counterparties.

Typical market conventions include:

| Payment Frequency | Payments per Year |
|-------------------|------------------:|
| Annual            | 1 |
| Semi-Annual       | 2 |
| Quarterly         | 4 |
| Monthly           | 12 |

The payment frequency determines the length of each contractual accrual period.

### Accrual Period

An accrual period is the time interval over which interest accumulates.

For example, an annual swap beginning on 1 January 2026 will have its first accrual period extending until 1 January 2027.

### Payment Date

The payment date is the date on which the coupon calculated over the accrual period is exchanged between counterparties.

Under the simplifying assumptions adopted in this project, the payment date is identical to the accrual end date.

## Example Timeline

Consider a five-year vanilla Interest Rate Swap with annual payments.

The schedule generation engine converts the contractual information into five payment periods.

Effective Date                                              Maturity Date

01-Jan-2026

│──────────│──────────│──────────│──────────│──────────│

01-Jan-27  01-Jan-28  01-Jan-29  01-Jan-30  01-Jan-31   

Each interval represents a single contractual payment period.

Every payment period contains:

- Accrual Start Date
- Accrual End Date
- Payment Date
- Accrual Length

These periods form the contractual timeline that will subsequently be transformed into coupon cash flows.

## Software Architecture

The schedule generation engine has been designed using small immutable domain objects with clearly separated responsibilities.

Instead of representing schedules as lists of dates, the project models the financial concepts directly through dedicated classes.

In [ ]:
InterestRateSwap
        │
        ▼
ScheduleGenerator
        │
        ▼
Schedule
        │
        ▼
PaymentPeriod

Each component has a single responsibility.

### InterestRateSwap

Represents the contractual agreement between two counterparties.

It combines the trade information together with the fixed and floating legs.

### ScheduleGenerator

A stateless service responsible for constructing payment schedules from the contractual terms of the swap.

### Schedule

An immutable collection of payment periods.

It provides convenient iteration, indexing, and access to the first and last payment periods.

### PaymentPeriod

Represents one contractual accrual period and stores:

- Accrual Start Date
- Accrual End Date
- Payment Date
- Accrual Length

## Design Principles

Several design principles guided the implementation of the schedule generation engine.

### Immutability

Financial contracts should not change after construction.

All domain objects are therefore implemented as immutable dataclasses.

### Separation of Responsibilities

The Schedule object stores payment periods.

The ScheduleGenerator creates payment periods.

Separating creation from storage keeps both components focused on a single responsibility.

### Explicit Domain Objects

Instead of using tuples or dictionaries to represent payment periods, the project introduces dedicated domain classes.

This approach improves readability, maintainability, and type safety while closely reflecting the terminology used in financial institutions.

### Incremental Architecture

The pricing library is intentionally developed in layers.

Each completed module becomes the foundation for the next, allowing the codebase to grow without introducing unnecessary complexity.

## Implementation Overview

The schedule generation module is implemented using three core classes, each with a well-defined responsibility.

Rather than combining all functionality into a single class, the implementation separates the representation of payment periods, the collection of periods, and the logic responsible for generating them.

This separation follows object-oriented design principles and makes the implementation easier to extend in future modules.

### PaymentPeriod

`PaymentPeriod` represents a single contractual accrual period.

Each instance stores the key dates required for coupon calculation:

- Accrual Start Date
- Accrual End Date
- Payment Date

The class also provides a computed property for the accrual length measured in days.

Since contractual periods should never change after creation, the class is implemented as an immutable dataclass.

@dataclass(frozen=True, slots=True)
class PaymentPeriod:
    ...

The `accrual_days` property is calculated dynamically from the start and end dates, ensuring that the value always remains consistent with the stored dates.

### Schedule

A Schedule represents the complete sequence of contractual payment periods for an Interest Rate Swap.

Internally, the payment periods are stored as an immutable tuple.

The Schedule class provides convenience methods that simplify downstream pricing logic, including:

- Iteration over payment periods
- Random access using indexing
- Number of payment periods
- First payment period
- Last payment period
- Overall schedule start date
- Overall schedule end date

Using a dedicated Schedule object provides a richer domain model than returning a plain Python list.

@dataclass(frozen=True, slots=True)
class Schedule:
    ...

### ScheduleGenerator

Schedule generation is performed by the `ScheduleGenerator` service.

Its responsibility is to transform an Interest Rate Swap into a sequence of contractual payment periods.

The generator determines the payment interval from the payment frequency, constructs each payment period in chronological order, and returns an immutable Schedule object.

class ScheduleGenerator:
    ...

## Schedule Generation Algorithm

The schedule generation process follows a straightforward sequence of steps.

Receive Interest Rate Swap
            │
            ▼
Read Effective Date
            │
            ▼
Read Maturity Date
            │
            ▼
Determine Payment Frequency
            │
            ▼
Calculate Interval Length
            │
            ▼
Generate Payment Periods
            │
            ▼
Store Periods in Schedule
            │
            ▼
Return Immutable Schedule

For an annual swap, the interval length is one year.

For a quarterly swap, the interval length is three months.

The implementation calculates this interval directly from the selected payment frequency, making the algorithm applicable to multiple payment conventions without changing the overall design.

## Simplifying Assumptions

To keep the initial implementation focused, several simplifying assumptions have been adopted.

- Payment date equals accrual end date.
- Business day adjustments are not applied.
- Holiday calendars are not considered.
- Stub periods are not supported.
- All periods follow a regular payment frequency.
- The schedule is generated using calendar dates only.

These assumptions allow the project to focus on the core pricing architecture before introducing market conventions in later iterations.

## Demonstration

Having designed the schedule generation module, we now demonstrate its usage by constructing a five-year vanilla Interest Rate Swap and generating its contractual payment schedule.

The example below illustrates how the scheduling engine transforms the contractual terms of a swap into a sequence of immutable payment periods.

In [2]:
from datetime import date

import pandas as pd

from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PaymentFrequency,
    PayReceive,
)
from src.products.trade import Trade
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap
from src.schedule.schedule_generator import ScheduleGenerator

In [3]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC Bank",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2031, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
    fixed_rate=0.05,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
    spread=0.0,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

In [4]:
generator = ScheduleGenerator()

schedule = generator.generate(swap)

In [5]:
schedule_df = pd.DataFrame(
    [
        {
            "Period": i,
            "Accrual Start": period.accrual_start,
            "Accrual End": period.accrual_end,
            "Payment Date": period.payment_date,
            "Accrual Days": period.accrual_days,
        }
        for i, period in enumerate(schedule, start=1)
    ]
)

schedule_df

,Period,Accrual Start,Accrual End,Payment Date,Accrual Days
0,1,2026-01-01,2027-01-01,2027-01-01,365
1,2,2027-01-01,2028-01-01,2028-01-01,365
2,3,2028-01-01,2029-01-01,2029-01-01,366
3,4,2029-01-01,2030-01-01,2030-01-01,365
4,5,2030-01-01,2031-01-01,2031-01-01,365


In [6]:
assert len(schedule) == 5
assert schedule.start_date == date(2026, 1, 1)
assert schedule.end_date == date(2031, 1, 1)

print("✓ Schedule generated successfully.")

✓ Schedule generated successfully.


## Conclusion

In this notebook, we implemented the schedule generation engine for a vanilla Interest Rate Swap.

Starting from the contractual trade definition, the engine generates an immutable sequence of payment periods that accurately represents the timeline of the swap. This schedule provides the foundation for downstream modules, including cash flow generation, discounting, pricing, and risk analytics.

By separating contractual representation from schedule generation, the implementation remains modular, extensible, and aligned with object-oriented design principles commonly used in financial software.

## Key Takeaways

In this module, we learned:

- Schedule generation transforms contractual trade information into a sequence of payment periods.
- Each payment period captures the dates required for future coupon calculations.
- The implementation separates domain objects from generation logic, resulting in a clean and maintainable design.
- The generated `Schedule` object becomes the foundation for all subsequent valuation and risk calculations.

### Next Module

In the next module, we will build the **Cash Flow Generation** engine.

Using the payment schedule created in this notebook, we will calculate the fixed and floating coupon cash flows associated with each payment period.